# Dataset 3: Wholesale Price and MSP Data Profiling & Rescue Notebook


In [25]:
import os
import json
import re
import pandas as pd
import numpy as np


In [26]:
# Load Raw Price & MSP JSON Dataset
raw_json_path = "../data/raw/track3_price_and_msp.json"
with open(raw_json_path, 'r', encoding='utf-8') as f:
    price_json = json.load(f)
df_raw = pd.DataFrame(price_json)

In [27]:
df_raw.head(10)

,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
0,PR001650,2026/07/26,MANDI005,None,Kapas,6850.985817820893,"₹7,570.17",7210.58,"₹6,620.00"
1,PR000246,2026-07-17,MANDI028,Fatehabad,कपास,"₹6,944.79",7654.18,"Rs. 7,299",
2,PR010091,09.01.2026,MANDI011,Jalandhar,Dhaan,"Rs. 1,857",2013.18,"₹1,935.06","INR 2,183"
3,PR000982,2026/05/24,M012,Ferozepur,corn,"₹1,873.34","Rs. 1,986","Rs. 1,930","Rs. 2,090"
4,PR001708,18/08/2026,013,Karnal,Cotton,"5,878.62/-",6574.283051769212,6226.45,"Rs. 6,620"
5,PR004312,01-24-2026,MANDI001,Patiala,Kapas,,"₹7,324.58",,"6,620.00/-"
6,PR000288,08-Aug-2026,M031,Hisar,Kanak,"₹2,270.14",2823.4429693584902,"2,546.79/-","₹2,275.00"
7,PR002536,2026-01-06,M036,Kurukshetra,WHEAT,"₹1,961.63","Rs. 2,340","INR 2,151",2275
8,PR009427,12-Aug-2026,MANDI-050,,Gehun,2277.53,"INR 2,458","₹2,367.64","2,275.00/-"
9,PR003418,09.02.2026,M011,Ambala,गन्ना,"₹3,156.94",3890.02,,"Rs. 3,500"


In [28]:
df_raw.shape

(12000, 9)

In [29]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   record_id    12000 non-null  object
 1   date         12000 non-null  object
 2   mandi_id     10765 non-null  object
 3   district     11227 non-null  object
 4   crop_name    12000 non-null  object
 5   min_price    12000 non-null  object
 6   max_price    12000 non-null  object
 7   modal_price  12000 non-null  object
 8   msp          12000 non-null  object
dtypes: object(9)
memory usage: 843.9+ KB


### Step 1: Raw JSON Data Profiling and Messiness Audit

**Problem Strategy**:
1. `track3_price_and_msp.json` contains 12,000 wholesale trading records with prices and Minimum Support Prices (MSP).
2. We inspect missing location keys (`mandi_id`, `district`), messy string prices containing currency symbols (`₹`, `Rs.`, `INR`, `,`), mixed date strings (`2026/07/26`, `08-Aug-2026`), and crop name aliases.

In [30]:
# Check Raw Dimensions & Missing Values
initial_records = len(df_raw)
missing_mandi_before = df_raw['mandi_id'].isnull().sum()
missing_district_before = df_raw['district'].isnull().sum()

In [31]:
print("Before Cleaning Price & MSP Dataset Scan:- ")
print("Total Raw JSON Records Loaded:", initial_records)
print("Missing mandi_id entries:", missing_mandi_before)
print("Missing district entries:", missing_district_before)

Before Cleaning Price & MSP Dataset Scan:- 
Total Raw JSON Records Loaded: 12000
Missing mandi_id entries: 1235
Missing district entries: 773


In [32]:
# Check for currency symbols in the 'modal_price' column
currency_symbols_count = df_raw['modal_price'].apply(
    lambda x: bool(re.search(r'Rs|₹|INR', str(x)))
).sum()

In [33]:
commas_count = df_raw['modal_price'].apply(
    lambda x: ',' in str(x)
).sum()

In [34]:
print("Modal price records with currency symbols (₹/Rs/INR):", currency_symbols_count)
print("Modal price records with commas:", commas_count)


Modal price records with currency symbols (₹/Rs/INR): 6044
Modal price records with commas: 7176


In [35]:
print("First 10 Raw Price JSON Records:- ")
df_raw.head(10)

First 10 Raw Price JSON Records:- 


,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
0,PR001650,2026/07/26,MANDI005,None,Kapas,6850.985817820893,"₹7,570.17",7210.58,"₹6,620.00"
1,PR000246,2026-07-17,MANDI028,Fatehabad,कपास,"₹6,944.79",7654.18,"Rs. 7,299",
2,PR010091,09.01.2026,MANDI011,Jalandhar,Dhaan,"Rs. 1,857",2013.18,"₹1,935.06","INR 2,183"
3,PR000982,2026/05/24,M012,Ferozepur,corn,"₹1,873.34","Rs. 1,986","Rs. 1,930","Rs. 2,090"
4,PR001708,18/08/2026,013,Karnal,Cotton,"5,878.62/-",6574.283051769212,6226.45,"Rs. 6,620"
5,PR004312,01-24-2026,MANDI001,Patiala,Kapas,,"₹7,324.58",,"6,620.00/-"
6,PR000288,08-Aug-2026,M031,Hisar,Kanak,"₹2,270.14",2823.4429693584902,"2,546.79/-","₹2,275.00"
7,PR002536,2026-01-06,M036,Kurukshetra,WHEAT,"₹1,961.63","Rs. 2,340","INR 2,151",2275
8,PR009427,12-Aug-2026,MANDI-050,,Gehun,2277.53,"INR 2,458","₹2,367.64","2,275.00/-"
9,PR003418,09.02.2026,M011,Ambala,गन्ना,"₹3,156.94",3890.02,,"Rs. 3,500"


### Step 2: Currency Symbol Cleaning and Numeric Price Parsing

**Problem Strategy**:
1. Raw price columns (`min_price`, `max_price`, `modal_price`, `msp`) contain non-numeric currency symbols (`₹`, `Rs.`, `INR`), commas (`,`), and trailing slashes (`/-`). Standard numerical functions fail on these text strings.
2. We parse each price field using regex `re.sub(r'[^\d.]', '', str(val))` to extract double-precision floats.


In [37]:
# Robust price parsing function (strips Rs., INR, ₹, /- prefixes before extracting decimal float)
def clean_price_value(val):
    if pd.isna(val) or val is None:
        return np.nan
    s_val = str(val).strip().replace(',', '')
    if not s_val or s_val.lower() == 'nan':
        return np.nan
    
    # Strip currency prefixes (Rs., INR, ₹, /-)
    s_val = re.sub(r'(?i)Rs\.?|INR|₹|/-', '', s_val).strip()
    
    # Extract number with decimal point
    num_match = re.search(r'([-+]?\d*\.?\d+)', s_val)
    if num_match:
        try:
            return float(num_match.group(1))
        except:
            return np.nan
    return np.nan


In [39]:
# Before Checking Price Parsing
print("Before Price Parsing Sample Raw Values:- ")
print(df_raw[['min_price', 'max_price', 'modal_price', 'msp']].head(10))


Before Price Parsing Sample Raw Values:- 
           min_price           max_price modal_price         msp
0  6850.985817820893           ₹7,570.17     7210.58   ₹6,620.00
1          ₹6,944.79             7654.18   Rs. 7,299            
2          Rs. 1,857             2013.18   ₹1,935.06   INR 2,183
3          ₹1,873.34           Rs. 1,986   Rs. 1,930   Rs. 2,090
4         5,878.62/-   6574.283051769212     6226.45   Rs. 6,620
5                              ₹7,324.58              6,620.00/-
6          ₹2,270.14  2823.4429693584902  2,546.79/-   ₹2,275.00
7          ₹1,961.63           Rs. 2,340   INR 2,151        2275
8            2277.53           INR 2,458   ₹2,367.64  2,275.00/-
9          ₹3,156.94             3890.02               Rs. 3,500


In [40]:

# Apply clean price parsing
df = df_raw.copy()
df['clean_min_price'] = df['min_price'].apply(clean_price_value)
df['clean_max_price'] = df['max_price'].apply(clean_price_value)
df['clean_modal_price'] = df['modal_price'].apply(clean_price_value)
df['clean_msp'] = df['msp'].apply(clean_price_value)


In [41]:
# After Comparison Verification
print("\nAfter Price Comparison:- ")
print(df[['modal_price', 'clean_modal_price', 'msp', 'clean_msp']].head(10))



After Price Comparison:- 
  modal_price  clean_modal_price         msp  clean_msp
0     7210.58            7210.58   ₹6,620.00     6620.0
1   Rs. 7,299            7299.00                    NaN
2   ₹1,935.06            1935.06   INR 2,183     2183.0
3   Rs. 1,930            1930.00   Rs. 2,090     2090.0
4     6226.45            6226.45   Rs. 6,620     6620.0
5                            NaN  6,620.00/-     6620.0
6  2,546.79/-            2546.79   ₹2,275.00     2275.0
7   INR 2,151            2151.00        2275     2275.0
8   ₹2,367.64            2367.64  2,275.00/-     2275.0
9                            NaN   Rs. 3,500     3500.0


In [42]:
# Cleaned Summary Stats
print("\nCleaned Price Summary Statistics:- ")
print(df[['clean_min_price', 'clean_max_price', 'clean_modal_price', 'clean_msp']].describe().T)



Cleaned Price Summary Statistics:- 
                     count         mean          std      min        25%  \
clean_min_price    11402.0  3535.979492  1721.091287  1777.33  2069.0550   
clean_max_price    11402.0  4064.619881  1994.553062  1872.20  2372.0000   
clean_modal_price  11390.0  3797.821686  1853.615307  1827.41  2222.7325   
clean_msp           9623.0  3719.780734  1794.947140  2090.00  2183.0000   

                           50%        75%      max  
clean_min_price    2980.000000  5391.5150  6950.69  
clean_max_price    3196.865000  6157.0525  8643.54  
clean_modal_price  3083.060207  5780.0000  7790.00  
clean_msp          2275.000000  5650.0000  6620.00  


### Step 3: Raw Inspection of Mandi ID, Crop Names & Trading Dates
**Problem Strategy**:
Before standardizing or mapping, we perform a complete profiling scan of:
1. `mandi_id`: Identify unstandardized formats (`M012`, `013`, `MANDI-050`) and missing count.
2. `crop_name`: List all raw English, Hindi, and vernacular crop variants to build a 100% accurate mapping dictionary.
3. `date`: Scan raw date string formats (`2026/07/26`, `09.01.2026`, `18/08/2026`, `08-Aug-2026`).


In [44]:
# Raw Mandi ID, Crop Names, and Trading Dates Check
print("RAW MANDI ID PROFILE:- ")
print("Missing mandi_id count:", df_raw['mandi_id'].isnull().sum())
print("Sample raw mandi_id values:")
print(df_raw['mandi_id'].dropna().head(10).tolist())
print("RAW CROP NAMES PROFILE:- ")
print("Total unique raw crop variants:", df_raw['crop_name'].nunique())
print("Raw Crop Name Value Counts:- ")
print(df_raw['crop_name'].value_counts())
print("RAW TRADING DATES SAMPLE:- ")
print("Sample raw date strings:- ")
print(df_raw['date'].head(10).tolist())


RAW MANDI ID PROFILE:- 
Missing mandi_id count: 1235
Sample raw mandi_id values:
['MANDI005', 'MANDI028', 'MANDI011', 'M012', '013', 'MANDI001', 'M031', 'M036', 'MANDI-050', 'M011']
RAW CROP NAMES PROFILE:- 
Total unique raw crop variants: 36
Raw Crop Name Value Counts:- 
crop_name
Kapas        442
Ganna        442
गन्ना        415
Sarso        414
Narma        412
सरसों        412
sugarcane    404
Sarson       403
Sugarcane    394
Cotton       394
mustard      394
Mustard      388
cotton       386
Maize        365
कपास         362
Ganne        359
मक्का        356
corn         350
Corn         340
Makka        327
Makki        326
गेहूं        299
WHEAT        294
Wheat        292
wheat        291
GEHUN        273
Dhaan        271
Gehun        268
Kanak        266
Basmati      256
paddy        243
Paddy        242
धान          240
Chawal       233
चावल         225
Rice         222
Name: count, dtype: int64
RAW TRADING DATES SAMPLE:- 
Sample raw date strings:- 
['2026/07/26', '2026-07-